In [3]:
import warnings
import os
import numpy as np
import pandas as pd
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
import joblib
import optuna
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error

# --- Global Constants ---
PREDS_PATH = './model_predictions/'
DATA_PATH = './'
MODELS_SAVE_PATH = './final_models/'
PREVIOUS_BEST_SCORE = 283164.67 # The score from the champion CatBoost-only meta-model
N_OPTUNA_TRIALS = 30
RANDOM_STATE = 42

# --- Winkler Score Helper Function ---
def winkler_score(y_true, lower, upper, alpha=0.1):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (lower - y_true) * (2 / alpha), 0)
    penalty_upper = np.where(y_true > upper, (y_true - upper) * (2 / alpha), 0)
    return np.mean(width + penalty_lower + penalty_upper)

warnings.filterwarnings('ignore')
print("Libraries and helper functions loaded successfully.")

Libraries and helper functions loaded successfully.


In [4]:
print("\n--- Loading all necessary data and recreating meta-features ---")

# --- Load Base Predictions and Raw Data ---
try:
    oof_xgb_preds = np.load(f'{PREDS_PATH}oof_xgb_preds.npy')
    test_xgb_preds = np.load(f'{PREDS_PATH}test_xgb_preds.npy')
    oof_cb_preds = np.load(f'{PREDS_PATH}oof_cb_preds.npy')
    test_cb_preds = np.load(f'{PREDS_PATH}test_cb_preds.npy')
    oof_nn_preds = np.load(f'{PREDS_PATH}oof_nn_preds.npy')
    test_nn_preds = np.load(f'{PREDS_PATH}test_nn_preds.npy')
    y_true = pd.read_csv(DATA_PATH + 'dataset.csv')['sale_price']
    df_train_raw = pd.read_csv(DATA_PATH + 'dataset.csv')
    df_test_raw = pd.read_csv(DATA_PATH + 'test.csv')
except FileNotFoundError as e:
    print(f"\nERROR: Could not find a required file. {e}")

# --- Recreate Super-Ensemble Mean ---
best_mean_weights = [0.4100, 0.4779, 0.1121] # From original notebook
oof_ensemble_mean = np.dot(np.vstack([oof_xgb_preds, oof_cb_preds, oof_nn_preds]).T, best_mean_weights)
test_ensemble_mean = np.dot(np.vstack([test_xgb_preds, test_cb_preds, test_nn_preds]).T, best_mean_weights)

# --- Recreate Context Features ---
def create_context_features(df):
    df = df.rename(columns={'latitude': 'lat', 'longitude': 'long', 'year_built': 'yr_built', 'year_reno':'yr_renovated'})
    X_context = df[['grade', 'sqft', 'lat', 'long', 'yr_built', 'yr_renovated']].fillna(0)
    X_context['property_age'] = 2025 - X_context['yr_built']
    return X_context

X_context_train = create_context_features(df_train_raw)
X_context_test = create_context_features(df_test_raw)

# --- Combine into Final Enhanced Meta-Feature Sets ---
X_meta_train_enhanced = pd.concat([pd.DataFrame({'xgb_pred': oof_xgb_preds, 'cb_pred': oof_cb_preds, 'nn_pred': oof_nn_preds, 'ensemble_mean': oof_ensemble_mean}), X_context_train.reset_index(drop=True)], axis=1)
X_meta_test_enhanced = pd.concat([pd.DataFrame({'xgb_pred': test_xgb_preds, 'cb_pred': test_cb_preds, 'nn_pred': test_nn_preds, 'ensemble_mean': test_ensemble_mean}), X_context_test.reset_index(drop=True)], axis=1)

print("All data loaded and meta-features recreated successfully.")


--- Loading all necessary data and recreating meta-features ---
All data loaded and meta-features recreated successfully.


In [8]:
print("\n--- STAGE 1: Tuning and Training the new LightGBM Meta-Model ---")

# Create a validation split for tuning
from sklearn.model_selection import train_test_split
X_meta_train_tune, X_meta_val_tune, y_meta_train_tune, y_meta_val_tune = train_test_split(
    X_meta_train_enhanced, y_true, test_size=0.25, random_state=RANDOM_STATE
)
N_OPTUNA_TRIALS = 100
# --- Define Optuna Objective ---
def objective_winkler_meta_lgb(trial):
    params = {
        'objective': 'quantile',
        'metric': 'quantile',
        'n_estimators': 2500,  # Increased for better convergence
        'verbosity': -1,
        'n_jobs': -1,
        'learning_rate': trial.suggest_float('learning_rate', 0.016, 0.022, log=True),  # Narrowed around best 0.018-0.019
        'num_leaves': trial.suggest_int('num_leaves', 40, 55),  # Optimal range from top trials
        'max_depth': trial.suggest_int('max_depth', 9, 10),  # Best values consistently at depth 9-10
        'subsample': trial.suggest_float('subsample', 0.65, 0.75),  # Tightened around best 0.68-0.73
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.59, 0.65),  # Focused on 0.60-0.65
        'reg_alpha': trial.suggest_float('reg_alpha', 12.0, 16.0),  # Higher regularization optimal
        'reg_lambda': trial.suggest_float('reg_lambda', 9.0, 13.0),  # Tightened around best 10-11
        'min_child_samples': trial.suggest_int('min_child_samples', 18, 25)  # Added new key parameter
    }
    params['alpha'] = 0.05
    model_lower = lgb.LGBMRegressor(**params)
    model_lower.fit(X_meta_train_tune, y_meta_train_tune, 
                   eval_set=[(X_meta_val_tune, y_meta_val_tune)], 
                   callbacks=[lgb.early_stopping(50, verbose=False)])  # Increased patience
    
    params['alpha'] = 0.95
    model_upper = lgb.LGBMRegressor(**params)
    model_upper.fit(X_meta_train_tune, y_meta_train_tune, 
                   eval_set=[(X_meta_val_tune, y_meta_val_tune)], 
                   callbacks=[lgb.early_stopping(50, verbose=False)])
    
    lower_preds = model_lower.predict(X_meta_val_tune)
    upper_preds = model_upper.predict(X_meta_val_tune)
    return winkler_score(y_meta_val_tune, lower_preds, upper_preds)

# --- Run Tuning ---
print(f"Running Optuna study for {N_OPTUNA_TRIALS} trials for LightGBM meta-model...")
study_meta_lgb = optuna.create_study(direction='minimize')
study_meta_lgb.optimize(objective_winkler_meta_lgb, n_trials=N_OPTUNA_TRIALS)
best_params_meta_lgb = study_meta_lgb.best_params
print(f"LightGBM Tuning Complete. Best score found: {study_meta_lgb.best_value:,.2f}")

# --- Train and Save Final LGBM Models ---
print("\nTraining and saving final LightGBM models...")
final_params_lgb = best_params_meta_lgb.copy()
final_params_lgb['objective'], final_params_lgb['metric'] = 'quantile', 'quantile'
# Train Lower
final_params_lgb['alpha'] = 0.05
model_lgb_lower = lgb.LGBMRegressor(**final_params_lgb, n_estimators=2500).fit(X_meta_train_enhanced, y_true)
# Train Upper
final_params_lgb['alpha'] = 0.95
model_lgb_upper = lgb.LGBMRegressor(**final_params_lgb, n_estimators=2500).fit(X_meta_train_enhanced, y_true)
# Save
os.makedirs(MODELS_SAVE_PATH, exist_ok=True)
joblib.dump(model_lgb_lower, os.path.join(MODELS_SAVE_PATH, 'meta_model_lgbm_lower.joblib'))
joblib.dump(model_lgb_upper, os.path.join(MODELS_SAVE_PATH, 'meta_model_lgbm_upper.joblib'))
print("Final LightGBM meta-models trained and saved.")

[I 2025-07-17 17:22:56,736] A new study created in memory with name: no-name-e692d888-cfbf-49d1-9df5-219e4b3eb82b



--- STAGE 1: Tuning and Training the new LightGBM Meta-Model ---
Running Optuna study for 100 trials for LightGBM meta-model...


[I 2025-07-17 17:23:09,227] Trial 0 finished with value: 314790.416748449 and parameters: {'learning_rate': 0.0177141147446419, 'num_leaves': 49, 'max_depth': 10, 'subsample': 0.6563299780567962, 'colsample_bytree': 0.6337173403598146, 'reg_alpha': 13.383946146292836, 'reg_lambda': 9.281861235269933, 'min_child_samples': 19}. Best is trial 0 with value: 314790.416748449.
[I 2025-07-17 17:23:18,022] Trial 1 finished with value: 314766.21287524025 and parameters: {'learning_rate': 0.020831770706445124, 'num_leaves': 54, 'max_depth': 10, 'subsample': 0.7339039040924266, 'colsample_bytree': 0.6179263279208969, 'reg_alpha': 12.389377422564934, 'reg_lambda': 9.806851250268261, 'min_child_samples': 24}. Best is trial 1 with value: 314766.21287524025.
[I 2025-07-17 17:23:28,875] Trial 2 finished with value: 314359.6591756019 and parameters: {'learning_rate': 0.019668393257999798, 'num_leaves': 53, 'max_depth': 9, 'subsample': 0.7469895632676846, 'colsample_bytree': 0.6183231571205701, 'reg_alp

LightGBM Tuning Complete. Best score found: 314,108.23

Training and saving final LightGBM models...
Final LightGBM meta-models trained and saved.


In [11]:
print("\n--- STAGE 2: Final Blending, Analysis, and Submission ---")

# --- Load the pre-trained champion models ---
try:
    print("Loading pre-trained XGBoost and CatBoost meta-models...")
    model_xgb_lower = joblib.load(os.path.join(MODELS_SAVE_PATH, 'meta_model_xg_final_lower.joblib'))
    model_xgb_upper = joblib.load(os.path.join(MODELS_SAVE_PATH, 'meta_model_xg_final_upper.joblib'))
    model_cb_lower = joblib.load(os.path.join(MODELS_SAVE_PATH, 'meta_model_catboost_lower.joblib'))
    model_cb_upper = joblib.load(os.path.join(MODELS_SAVE_PATH, 'meta_model_catboost_upper.joblib'))
    print("Existing champion models loaded.")
except FileNotFoundError as e:
    print(f"\nFATAL ERROR: Could not find a previously saved meta-model. Please run the full training notebook first. {e}")

# --- Generate OOF predictions from all 3 meta-models ---
print("\nGenerating OOF predictions from all models for blending...")
oof_xgb_lower = model_xgb_lower.predict(X_meta_train_enhanced)
oof_xgb_upper = model_xgb_upper.predict(X_meta_train_enhanced)
oof_cb_lower = model_cb_lower.predict(X_meta_train_enhanced)
oof_cb_upper = model_cb_upper.predict(X_meta_train_enhanced)
oof_lgb_lower = model_lgb_lower.predict(X_meta_train_enhanced)
oof_lgb_upper = model_lgb_upper.predict(X_meta_train_enhanced)

# --- Find the Optimal 3-Model Blend ---
print("\nSearching for the optimal 3-model blend...")
def get_winkler_from_3_blend(weights):
    w_xgb, w_cb, w_lgb = weights[0], weights[1], weights[2]
    final_lower = w_xgb * oof_xgb_lower + w_cb * oof_cb_lower + w_lgb * oof_lgb_lower
    final_upper = w_xgb * oof_xgb_upper + w_cb * oof_cb_upper + w_lgb * oof_lgb_upper
    return winkler_score(y_true, final_lower, final_upper)

result_blend = minimize(get_winkler_from_3_blend, [1/3]*3, method='SLSQP', bounds=[(0,1)]*3, constraints=({'type': 'eq', 'fun': lambda w: 1 - sum(w)}))
best_weights = result_blend.x
best_final_score = result_blend.fun

# --- Final Showdown ---
print("\n" + "="*60)
print("     THE THREE-CHAMPION META-MODEL PIPELINE: FINAL SHOWDOWN")
print("="*60)
print(f"Previous Best (CatBoost-only)     : ${PREVIOUS_BEST_SCORE:,.2f}")
print(f"New 3-MODEL ENSEMBLE Score        : ${best_final_score:,.2f}")
print(f" (Optimal Weights: XGB={best_weights[0]:.4f}, CB={best_weights[1]:.4f}, LGBM={best_weights[2]:.4f})")
print("="*60)


--- STAGE 2: Final Blending, Analysis, and Submission ---
Loading pre-trained XGBoost and CatBoost meta-models...
Existing champion models loaded.

Generating OOF predictions from all models for blending...

Searching for the optimal 3-model blend...

     THE THREE-CHAMPION META-MODEL PIPELINE: FINAL SHOWDOWN
Previous Best (CatBoost-only)     : $283,164.67
New 3-MODEL ENSEMBLE Score        : $282,946.15
 (Optimal Weights: XGB=0.0000, CB=0.8045, LGBM=0.1955)


In [12]:
# --- Create Final Submission File ---
print("\nCreating final submission file...")
test_xgb_lower = model_xgb_lower.predict(X_meta_test_enhanced)
test_xgb_upper = model_xgb_upper.predict(X_meta_test_enhanced)
test_cb_lower = model_cb_lower.predict(X_meta_test_enhanced)
test_cb_upper = model_cb_upper.predict(X_meta_test_enhanced)
test_lgb_lower = model_lgb_lower.predict(X_meta_test_enhanced)
test_lgb_upper = model_lgb_upper.predict(X_meta_test_enhanced)

final_test_lower = best_weights[0] * test_xgb_lower + best_weights[1] * test_cb_lower + best_weights[2] * test_lgb_lower
final_test_upper = best_weights[0] * test_xgb_upper + best_weights[1] * test_cb_upper + best_weights[2] * test_lgb_upper

submission_df = pd.DataFrame({'id': df_test_raw['id'], 'pi_lower': final_test_lower, 'pi_upper': final_test_upper})
submission_df['pi_lower'] = submission_df['pi_lower'].clip(0, None)
submission_df['pi_upper'] = np.maximum(submission_df['pi_lower'], submission_df['pi_upper'])
submission_filename = f'submission_final_3_model_blend_{int(best_final_score)}.csv'
submission_df.to_csv(submission_filename, index=False)
print(f"\n'{submission_filename}' created successfully!")
display(submission_df.head())


Creating final submission file...

'submission_final_3_model_blend_282946.csv' created successfully!


,id,pi_lower,pi_upper
0,200000,830367.167344,1.066608e+06
1,200001,574447.369746,7.678097e+05
2,200002,458000.563628,6.331321e+05
3,200003,290861.444618,4.251686e+05
4,200004,360481.925638,7.101812e+05
